In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
from pathlib import Path
import pandas as pd
import re, pathlib
from datetime import datetime

In [2]:
# =============================================================================
# 1.  Directory layout – pathlib all the way
# =============================================================================
SCRIPT_DIR   = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = SCRIPT_DIR.parent          # edit if your notebook is elsewhere

DATA_DIR       = PROJECT_ROOT / "data/"
SIMULATION_DIR = DATA_DIR / "simulations/"          # folder with ATTRIBUTE_* and wide SSP file
TORNADO_SIM_DIR = SIMULATION_DIR /"data_for_LSU"
OUTPUT_DIR     = DATA_DIR / "output/"
OUTPUT_SUFFIX = datetime.today().strftime('%Y-%m-%d')

In [3]:
louisiana = pd.read_csv(TORNADO_SIM_DIR / "louisiana.csv")

In [4]:
louisiana

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_trns
0,0,louisiana,0,0,367983.8318,68239.85746,80.234375,79198.54166,6714.566582,1154589.900,...,9.980000,0.322352,2.821843,4.032755,0.0,39.360000,15.368289,106.770000,4.290000,32.920000
1,0,louisiana,1,0,362385.6368,67201.71394,79.013757,77993.68198,6612.416842,1137024.945,...,9.968923,0.319496,2.800291,4.092947,0.0,40.166298,14.997723,108.605133,3.731796,31.374749
2,0,louisiana,2,0,361096.1267,66962.58391,78.732596,77716.14990,6588.887271,1132978.965,...,9.489254,0.320059,2.788854,4.099808,0.0,38.639666,13.763035,80.719306,3.559992,31.843131
3,0,louisiana,3,0,359935.7476,66747.40026,78.479589,77466.40976,6567.713942,1129338.147,...,10.279229,0.321689,2.777719,4.162110,0.0,41.465412,14.851001,146.300301,3.841408,33.650089
4,0,louisiana,4,0,358773.5212,66531.87404,78.226180,77216.27204,6546.506906,1125691.534,...,9.983308,0.323218,2.772568,4.173150,0.0,37.807488,13.488061,146.740554,3.879013,34.227053
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2407,127127,louisiana,31,0,311979.6878,53280.76885,83.099337,74341.21831,6871.576034,1116082.754,...,2.290044,0.099194,1.610628,2.566150,0.0,9.504129,1.357871,6.354095,0.494434,17.846785
2408,127127,louisiana,32,0,310670.5663,53049.87938,83.075690,74040.77745,6869.511396,1114713.368,...,2.285685,0.087899,1.527726,2.529939,0.0,9.783970,1.280190,4.725632,0.364936,17.059667
2409,127127,louisiana,33,0,309356.4650,52821.26235,83.037733,73734.23638,6866.309713,1113193.917,...,2.281655,0.076939,1.445208,2.497102,0.0,9.535228,1.030977,3.223424,0.239272,16.254621
2410,127127,louisiana,34,0,308044.9743,52594.98521,82.990728,73425.33726,6862.387927,1111581.984,...,2.277804,0.066352,1.363483,2.466951,0.0,9.696216,0.918047,1.841394,0.117569,15.427843


In [5]:
# 1) Filter out the base case
base_case = louisiana[louisiana['primary_id'] == 0].copy()


# Fugitive Emissions and CCS

In [6]:
# 2) Define your fuels and sectors
relevant_gases = ['n2o',
                  'sf6',
                  'c4f6',
                  'c4f8o',
                  'c2f6',
                  'c3f8',
                  'c6f14',
                  'c5f8',
                  'cc4f8']

sectors = ['chemicals',
           'electronics',
           'metals']



In [7]:
# 3) Initialize accumulators
emissions_avoided_by_sector = pd.DataFrame({'primary_id': base_case['primary_id'], 'time_period': base_case['time_period']}, index=base_case.index)

In [8]:
# 4) Industrial cost parameters
capex_chemicals_n2o = 20*0.2
opex_chemicals_n2o = 20*0.8
capex_electronics_sf6 = 40*0.5
opex_electronics_sf6 = 40*0.5
capex_electronics_c4f6 = 40*0.5
opex_electronics_c4f6 = 40*0.5
capex_electronics_c2f6 = 40*0.5
opex_electronics_c2f6 = 40*0.5
capex_electronics_c3f8 = 40*0.5
opex_electronics_c3f8 = 40*0.5
capex_electronics_c5f8 = 40*0.5
opex_electronics_c5f8 = 40*0.5
capex_electronics_cc4f8 = 40*0.5
opex_electronics_cc4f8 = 40*0.5
capex_metals_sf6 = 20*0.7
opex_metals_sf6 = 20*0.7
capex_metals_c2f6 = 20*0.7
opex_metals_c2f6 = 20*0.7

In [9]:
# 5) Loop over fuels and sectors
for gas in relevant_gases:
    for sector in sectors:
        abatement_cols = [c for c in base_case.columns
                if c.startswith(f'ef_ippu_tonne_{gas}_per_tonne_production_{sector}')]
   
        emissions_cols = [c for c in base_case.columns
                if c.startswith(f'emission_co2e_{gas}_ippu_production_{sector}')]

        if(len(abatement_cols)>0 and len(emissions_cols)>0):
            abatement_factor = base_case[abatement_cols[0]]
            emissions = base_case[emissions_cols[0]]

            if(abatement_factor.sum()>0):
                abatement_factor = abatement_factor[0]/abatement_factor
                emissions_avoided_by_sector[f'emissions_abated_{sector}_{gas}'] = (abatement_factor-1)*emissions
                emissions_avoided_by_sector[f'capex_emissions_abated_{sector}_{gas}'] = (abatement_factor-1)*emissions*globals()[f'capex_{sector}_{gas}']
                emissions_avoided_by_sector[f'opex_emissions_abated_{sector}_{gas}'] = (abatement_factor-1)*emissions*globals()[f'opex_{sector}_{gas}']



    


In [10]:
#ccs
emissions_avoided_by_sector['ccs'] = base_case['emission_co2e_subsector_total_ccsq'] - base_case['emission_co2e_subsector_total_ccsq'][0]
capex_ccs = 500*(0.96)**emissions_avoided_by_sector['time_period']
opex_ccs = 50*(0.96)**emissions_avoided_by_sector['time_period']
emissions_avoided_by_sector['capex_ccs'] = emissions_avoided_by_sector['ccs']*capex_ccs
emissions_avoided_by_sector['opex_ccs'] = emissions_avoided_by_sector['ccs']*opex_ccs


In [11]:
emissions_avoided_by_sector.to_csv(OUTPUT_DIR/f'fugitive_emissions_and_ccs_{OUTPUT_SUFFIX}.csv', index=False)

# SCOE

In [12]:
# 2) Define your fuels and sectors
relevant_fuels = ['solid_biomass', 
                    'coal',  
                    'diesel', 
                    'electricity',
                    'gasoline', 
                    'hydrocarbon_gas_liquids',
                    'hydrogen',
                    'kerosene',
                    'natural_gas']

sectors = ['commercial_municipal',
            'other_se',
            'residential']


In [13]:
# 3) Initialize accumulators
scoe_fuel_demand_by_sector = pd.DataFrame({'primary_id': base_case['primary_id'], 'time_period': base_case['time_period']}, index=base_case.index)

In [14]:
# 4) Industrial cost parameters
capex_industrial_electricity = 92666.6 * 21
capex_industrial_other       = 92666.6 * 12
opex_industrial_electricity  = 92666.6 * 2.5
opex_industrial_other        = 92666.6 * 4.5
capex_multiplier_efficiency = 5560000
opex_multiplier_efficiency = 0

In [15]:
# 5) Loop over fuels and sectors
for fuel in relevant_fuels:
    # find the efficiency column(s) for this fuel
    # find the demand column(s) for this fuel
    for sector in sectors:
        eff_cols = [c for c in base_case.columns
                if c.startswith(f'efficfactor_scoe_heat_energy_{sector}_{fuel}')]
        fuel_efficiency = base_case[eff_cols[0]]
        sector_dem_cols = [c for c in base_case.columns
                if (f'scalar_scoe_heat_energy_demand_{sector}' in c)]
        
        sector_fuel_fraction_cols = [c for c in base_case.columns
                if (f'frac_scoe_heat_energy_{sector}_{fuel}' in c)]

        if len(sector_fuel_fraction_cols)>0 and len(sector_dem_cols)>0:
            sector_total_demand = base_case[sector_dem_cols[0]]
            sector_fuel_fraction = base_case[sector_fuel_fraction_cols[0]]

            if (sector_fuel_fraction*sector_total_demand).sum()>0 or fuel=='electricity':
                sector_fuel_demand = sector_fuel_fraction*sector_total_demand
                scoe_fuel_demand_by_sector[f'energy_demand_{sector}_{fuel}'] = sector_fuel_demand
                if fuel=='electricity':
                    scoe_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = sector_fuel_demand*capex_industrial_electricity
                    scoe_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = sector_fuel_demand*opex_industrial_electricity
                else:
                    scoe_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = sector_fuel_demand*capex_industrial_other
                    scoe_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = sector_fuel_demand*opex_industrial_other
                sector_fuel_consumed = sector_fuel_demand/fuel_efficiency
                sector_fuel_consumed_baseline = sector_fuel_demand/fuel_efficiency.iloc[0]
                sector_change_in_fuel_consumed = sector_fuel_consumed_baseline-sector_fuel_consumed
                scoe_fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = sector_change_in_fuel_consumed
                scoe_fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = sector_change_in_fuel_consumed*capex_multiplier_efficiency
                scoe_fuel_demand_by_sector[f'efficiency_opex_{sector}_{fuel}'] = sector_change_in_fuel_consumed*opex_multiplier_efficiency

C:\Users\pkane\AppData\Local\Temp\ipykernel_18828\436411360.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  scoe_fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = sector_change_in_fuel_consumed
C:\Users\pkane\AppData\Local\Temp\ipykernel_18828\436411360.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  scoe_fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = sector_change_in_fuel_consumed*capex_multiplier_efficiency
C:\Users\pkane\AppData\Local\Temp\ipykernel_18828\436411360.py:33: Performance

In [16]:
scoe_fuel_demand_by_sector.to_csv(OUTPUT_DIR/f'scoe_{OUTPUT_SUFFIX}.csv', index=False)

# Industrial Energy and Fuel Efficiency

In [17]:
# 2) Define your fuels and sectors
relevant_fuels = ['biomass', 
                    'coal', 
                    'coke', 
                    'diesel', 
                    'electricity',
                    'furnace_gas',
                    'gasoline', 
                    'hydrocarbon_gas_liquids',
                    'hydrogen',
                    'kerosene',
                    'natural_gas',
                    'oil']

sectors = ['agriculture_and_livestock',
           'cement',
           'chemicals',
           'electronics',
           'glass',
           'lime_and_carbonite',
           'metals',
           'mining',
           'other_product_manufacturing',
           'paper',
           'plastic',
           'recycled_glass',
           'recycled_metals',
           'recycled_paper',
           'recycled_plastic',
           'recycled_rubber_and_leather',
           'recycled_textiles',
           'recycled_wood',
           'rubber_and_leather',
           'textiles',
           'wood']

In [18]:
# 3) Initialize accumulators
ind_fuel_demand_by_sector = pd.DataFrame({'primary_id': base_case['primary_id'], 'time_period': base_case['time_period']}, index=base_case.index)

In [19]:
# 4) Industrial cost parameters
capex_industrial_electricity = 92666.6 * 21
capex_industrial_other       = 92666.6 * 12
opex_industrial_electricity  = 92666.6 * 2.5
opex_industrial_other        = 92666.6 * 4.5
capex_multiplier_efficiency = 10000000
opex_multiplier_efficiency = 0

In [20]:
# 5) Loop over fuels and sectors
for fuel in relevant_fuels:
    # find the efficiency column(s) for this fuel
    eff_cols = [c for c in base_case.columns
                if c.startswith(f'efficfactor_enfu_industrial_energy_fuel_{fuel}')]
    fuel_efficiency = base_case[eff_cols[0]]
    # find the demand column(s) for this fuel
    for sector in sectors:
        sector_dem_cols = [c for c in base_case.columns
                if (f'energy_demand_inen_{sector}' in c)]
        
        sector_fuel_fraction_cols = [c for c in base_case.columns
                if (f'frac_inen_energy_{sector}_{fuel}' in c)]

        if len(sector_fuel_fraction_cols)>0 and len(sector_dem_cols)>0:
            sector_total_demand = base_case[sector_dem_cols[0]]
            sector_fuel_fraction = base_case[sector_fuel_fraction_cols[0]]

            if (sector_fuel_fraction*sector_total_demand).sum()>0 or fuel=='electricity':
                sector_fuel_demand = sector_fuel_fraction*sector_total_demand
                ind_fuel_demand_by_sector[f'energy_demand_{sector}_{fuel}'] = sector_fuel_demand
                if fuel=='electricity':
                    ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = sector_fuel_demand*capex_industrial_electricity
                    ind_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = sector_fuel_demand*opex_industrial_electricity
                else:
                    ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = sector_fuel_demand*capex_industrial_other
                    ind_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = sector_fuel_demand*opex_industrial_other
                sector_fuel_consumed = sector_fuel_demand/fuel_efficiency
                sector_fuel_consumed_baseline = sector_fuel_demand/fuel_efficiency.iloc[0]
                sector_change_in_fuel_consumed = sector_fuel_consumed_baseline-sector_fuel_consumed
                ind_fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = sector_change_in_fuel_consumed
                ind_fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = sector_change_in_fuel_consumed*capex_multiplier_efficiency
                ind_fuel_demand_by_sector[f'efficiency_opex_{sector}_{fuel}'] = sector_change_in_fuel_consumed*opex_multiplier_efficiency

C:\Users\pkane\AppData\Local\Temp\ipykernel_18828\1730409721.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ind_fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = sector_change_in_fuel_consumed
C:\Users\pkane\AppData\Local\Temp\ipykernel_18828\1730409721.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ind_fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = sector_change_in_fuel_consumed*capex_multiplier_efficiency
C:\Users\pkane\AppData\Local\Temp\ipykernel_18828\1730409721.py:33: Performanc

In [21]:
ind_fuel_demand_by_sector.to_csv(OUTPUT_DIR/f'industrial_energy_cost_{OUTPUT_SUFFIX}.csv', index=False)

# Transportation

In [22]:
dem_col = [c for c in base_case.columns 
           if 'energy_demand_enfu_subsector_total_pj_trns_fuel_electricity' in c][0]
eff_col = [c for c in base_case.columns 
           if c.startswith('elecfuelefficiency_trns_road_light')][0]

# 2) Compute saved volume (PJ) vs. baseline
elec_demand   = base_case[dem_col]
elec_eff      = base_case[eff_col]
vol_now       = elec_demand / elec_eff
vol_base      = elec_demand / elec_eff.iloc[0]
saved_pj      = vol_base - vol_now

# 3) Compute electricity cost at $880 000 per PJ saved
capex = saved_pj * 880_000

# 4) Build the output DataFrame
output_trns_elec = pd.DataFrame({
    'primary_id':                             base_case['primary_id'],
    'time_period':                            base_case['time_period'],
    'electricity_volume_saved_in_PJ':         saved_pj,
    'electricity_transportation_cost_$':      capex
}, index=base_case.index)

In [23]:
relevant_fuels = [
    'diesel',
    'gasoline',
    'hydrocarbon_gas_liquids',
    'hydrogen',
    'natural_gas'
]

# 4) Prepare accumulators
total_saved_volume = pd.Series(0.0, index=base_case.index)
demand_trns = pd.DataFrame({'time_period': base_case['time_period']},
                           index=base_case.index)

# 5) Loop & pattern-match per fuel
for fuel in relevant_fuels:
    # 5a) demand cols specific to this fuel
    dem_cols = [
        c for c in base_case.columns
        if f'energy_demand_enfu_subsector_total_pj_trns_fuel_{fuel}' in c
    ]
    # 5b) efficiency cols for this fuel
    eff_cols = [
        c for c in base_case.columns
        if f'fuelefficiency_trns_road_light_{fuel}' in c
    ]

    print(f"Fuel={fuel!r}: dem_cols={dem_cols}, eff_cols={eff_cols}")
    if not dem_cols or not eff_cols:
        print(f"  → skipping {fuel!r} (no matching columns)\n")
        continue

    fuel_demand     = base_case[dem_cols[0]]
    fuel_efficiency = base_case[eff_cols[0]]

    demand_trns[fuel] = fuel_demand

    vol_now  = fuel_demand / fuel_efficiency
    vol_base = fuel_demand / fuel_efficiency.iloc[0]
    total_saved_volume += (vol_base - vol_now)

# 6) Build output (with identifiers)
output_trns_non_elec = pd.DataFrame({
    'primary_id': base_case['primary_id'],
    'time_period': base_case['time_period'],
    'transportation_volume_saved_in_PJ': total_saved_volume
}, index=base_case.index)

# 7) Apply $880 000 per PJ
capex_multiplier_trns = 880_000
output_trns_non_elec['transportation_efficiency_capex'] = (
    output_trns_non_elec['transportation_volume_saved_in_PJ'] * capex_multiplier_trns
)
output_trns_non_elec['transportation_efficiency_opex'] = 0

Fuel='diesel': dem_cols=['energy_demand_enfu_subsector_total_pj_trns_fuel_diesel'], eff_cols=['fuelefficiency_trns_road_light_diesel_km_per_litre']
Fuel='gasoline': dem_cols=['energy_demand_enfu_subsector_total_pj_trns_fuel_gasoline'], eff_cols=['fuelefficiency_trns_road_light_gasoline_km_per_litre']
Fuel='hydrocarbon_gas_liquids': dem_cols=['energy_demand_enfu_subsector_total_pj_trns_fuel_hydrocarbon_gas_liquids'], eff_cols=['fuelefficiency_trns_road_light_hydrocarbon_gas_liquids_km_per_litre']
Fuel='hydrogen': dem_cols=['energy_demand_enfu_subsector_total_pj_trns_fuel_hydrogen'], eff_cols=['fuelefficiency_trns_road_light_hydrogen_km_per_litre']
Fuel='natural_gas': dem_cols=['energy_demand_enfu_subsector_total_pj_trns_fuel_natural_gas'], eff_cols=[]
  → skipping 'natural_gas' (no matching columns)



In [24]:
# 2) Detect the vehicle‐km column
dist_cols = [
    c for c in base_case.columns
    if c.startswith('vehicle_distance_traveled_trns_road_light')
]
if not dist_cols:
    raise KeyError("No vehicle_distance_traveled_trns_road_light_* column found")
dist_col = dist_cols[0]
vehicle_distance = base_case[dist_col]  # units: vkm

# 3) Define the three fuel‐mix fraction columns
relevant_fuels = ['diesel', 'electricity', 'gasoline']
frac_cols = {
    fuel: f'frac_trns_fuelmix_road_light_{fuel}'
    for fuel in relevant_fuels
}
for col in frac_cols.values():
    if col not in base_case.columns:
        raise KeyError(f"Missing fraction column: {col}")

# 4) Record the period-0 (baseline) shares
frac_baseline = {
    fuel: base_case[col].iloc[0]
    for fuel, col in frac_cols.items()
}

# 5) Compute how many vkm have switched fuels since the baseline
switched_from_diesel    = vehicle_distance * (frac_baseline['diesel']    - base_case[frac_cols['diesel']])
switched_from_gasoline  = vehicle_distance * (frac_baseline['gasoline']  - base_case[frac_cols['gasoline']])
switched_to_electricity = vehicle_distance * (base_case[frac_cols['electricity']] - frac_baseline['electricity'])

# 6) Use your lookup‐table multipliers (in $ per vkm)
#    * note the NEGATIVE sign for the electrification “cost” from the table
multipliers = {
    'electricity': {'cost': 0.039, 'saving':  0.012},
    'diesel':      {'cost':  0.000, 'saving':  0.000},
    'gasoline':    {'cost':  0.000, 'saving':  0.000},
}

# 7) Accumulate total cost & total savings
cost_series   = pd.Series(0.0, index=base_case.index)
saving_series = pd.Series(0.0, index=base_case.index)

for fuel in relevant_fuels:
    frac = base_case[frac_cols[fuel]]
    vkm  = vehicle_distance * frac
    cost_series   += vkm * multipliers[fuel]['cost']
    saving_series += vkm * multipliers[fuel]['saving']

# 8) Build the output table
output_fs = pd.DataFrame({
    'primary_id':                         base_case['primary_id'],
    'time_period':                        base_case['time_period'],
    'vehicle_distance_traveled_total_vkm': vehicle_distance,
    'switched_from_diesel_vkm':           switched_from_diesel,
    'switched_from_gasoline_vkm':         switched_from_gasoline,
    'switched_to_electricity_vkm':        switched_to_electricity,
    'fuel_switch_cost_$':                 cost_series,
    'fuel_switch_savings_$':              saving_series,
    'fuel_switch_net_cost_$':             cost_series - saving_series,
}, index=base_case.index)

In [25]:
# 2) Grab the heavy‐duty + public‐transit vehicle‐km
dist_patterns = [
    r"^vehicle_distance_traveled_trns_road_heavy_.*$",
    r"^vehicle_distance_traveled_trns_public.*",
]
dist_cols = [c for c in base_case.columns if any(re.match(p, c) for p in dist_patterns)]
if not dist_cols:
    raise KeyError("No heavy‐duty/public distance columns found")
vehicle_distance = base_case[dist_cols].sum(axis=1)

# 3) Define your fuel‐mix fraction columns & read off baseline shares
segments = ["freight", "regional"]
relevant_fuels = [
    "biofuels", "diesel", "electricity",
    "gasoline", "hydrocarbon_gas_liquids",
    "hydrogen", "natural_gas"
]
frac_cols = {
    fuel: [f"frac_trns_fuelmix_road_heavy_{seg}_{fuel}" for seg in segments]
    for fuel in relevant_fuels
}
# make sure they all exist
for fuel, cols in frac_cols.items():
    for col in cols:
        if col not in base_case.columns:
            raise KeyError(f"Missing fraction column: {col}")

# baseline share at t=0 (sum of freight+regional)
frac_baseline = {
    fuel: base_case[cols].sum(axis=1).iloc[0]
    for fuel, cols in frac_cols.items()
}

# 4) Define your $/vkm multipliers for each fuel
multipliers = {
    "biofuels":             {"cost": 0.00,   "saving": 0.00},
    "diesel":               {"cost": 0.00,   "saving": 0.00},
    "electricity":          {"cost": 0.042,  "saving": 0.020},
    "gasoline":             {"cost": 0.00,   "saving": 0.00},
    "hydrocarbon_gas_liquids": {"cost":0.00, "saving": 0.00},
    "hydrogen":             {"cost": 0.00,   "saving": 0.00},
    "natural_gas":          {"cost": 0.00,   "saving": 0.00},
}

# 5) Compute current & baseline costs/savings per fuel
cost_now   = pd.Series(0.0, index=base_case.index)
cost_base  = pd.Series(0.0, index=base_case.index)
saving_now  = pd.Series(0.0, index=base_case.index)
saving_base = pd.Series(0.0, index=base_case.index)

for fuel in relevant_fuels:
    # current total vkm on this fuel
    cur_frac = base_case[frac_cols[fuel]].sum(axis=1)
    vkm_now  = vehicle_distance * cur_frac
    # baseline total vkm on this fuel
    vkm_base = vehicle_distance * frac_baseline[fuel]
    # accumulate
    cost_now   += vkm_now  * multipliers[fuel]["cost"]
    cost_base  += vkm_base * multipliers[fuel]["cost"]
    saving_now  += vkm_now  * multipliers[fuel]["saving"]
    saving_base += vkm_base * multipliers[fuel]["saving"]

# 6) Difference from baseline
cost_series   = cost_now   - cost_base
saving_series = saving_now - saving_base
net_series    = cost_series - saving_series

# 7) Build output
output_hd = pd.DataFrame({
    "primary_id":                          base_case["primary_id"],
    "region":                              base_case["region"],
    "time_period":                         base_case["time_period"],
    "vehicle_distance_traveled_total_vkm": vehicle_distance,
    "fuel_switch_cost_$":                  cost_series,
    "fuel_switch_savings_$":               saving_series,
    "fuel_switch_net_cost_$":              net_series,
}, index=base_case.index)

In [26]:
# ——————————————————————————————————————————
# 2) Grab all rail electricity consumption (PJ) columns
# ——————————————————————————————————————————
rail_patterns = [r"^energy_consumption_trns_rail_.*_electricity$"]
rail_elec_cols = [
    c for c in base_case.columns
    if any(re.match(p, c) for p in rail_patterns)
]
if not rail_elec_cols:
    raise KeyError("No rail electricity consumption columns found")
# sum across any sub‐modes (freight, passenger, etc.)
rail_elec_PJ = base_case[rail_elec_cols].sum(axis=1)

# ——————————————————————————————————————————
# 3) Compute “switched to electricity” relative to baseline
# ——————————————————————————————————————————
baseline_elec = rail_elec_PJ.iloc[0]
switched_to_rail_elec_PJ = rail_elec_PJ - baseline_elec

# ——————————————————————————————————————————
# 4) Apply cost & saving multipliers ($ per PJ)
# ——————————————————————————————————————————
cost_mult   = 422_400_000   # $ per PJ
saving_mult =   2_377_710   # $ per PJ

cost_series   = switched_to_rail_elec_PJ * cost_mult
saving_series = switched_to_rail_elec_PJ * saving_mult
net_series    = cost_series - saving_series  # net capex

# ——————————————————————————————————————————
# 5) Build the output DataFrame
# ——————————————————————————————————————————
output_rail = pd.DataFrame({
    "primary_id":                          base_case["primary_id"],
    "region":                              base_case["region"],
    "time_period":                         base_case["time_period"],
    "rail_elec_consumption_PJ":            rail_elec_PJ,
    "switched_to_rail_elec_PJ":            switched_to_rail_elec_PJ,
    "rail_fuel_switch_cost_$":             cost_series,
    "rail_fuel_switch_saving_$":           saving_series,
    "rail_fuel_switch_net_cost_$":         net_series,
}, index=base_case.index)

In [27]:
output_trns_elec.to_csv(OUTPUT_DIR/f"transportation_electric_efficiency_cost_{OUTPUT_SUFFIX}.csv", index=False)
output_trns_non_elec.to_csv(OUTPUT_DIR/f"transportation_non_electric_efficiency_cost_{OUTPUT_SUFFIX}.csv", index=False)
output_fs.to_csv(OUTPUT_DIR/f'transportion_light_duty_fuel_switch_cost_{OUTPUT_SUFFIX}.csv', index=False)
output_hd.to_csv(OUTPUT_DIR /f"transportation_heavy_duty_fuel_switch_cost_{OUTPUT_SUFFIX}.csv", index=False)
output_rail.to_csv(OUTPUT_DIR /f"transportation_rail_fuel_switch_cost_{OUTPUT_SUFFIX}.csv", index=False)

# Energy Production

In [28]:
ID_COLS = ["primary_id", "time_period"]
value_cols = [c for c in base_case.columns if c not in ID_COLS]

long = base_case.melt(
    id_vars=ID_COLS, value_vars=value_cols,
    var_name="variable", value_name="value"
)

# ------------ identify three groups -----------------------------
is_capex = long["variable"].str.contains(
    r"nemomod_entc_discounted_capital_investment_", regex=True
)
is_opex = long["variable"].str.contains(
    r"nemomod_entc_discounted_operating_", regex=True
)
is_production = long["variable"].str.contains(
    r"nemomod_entc_annual_production_by_technology_", regex=True
)

capex_df      = long[is_capex].copy()
opex_df       = long[is_opex].copy()
production_df = long[is_production].copy()

# tag rows
capex_df["cost_type"] = "capex"
opex_df["cost_type"]  = "opex"

# ------------- helper to pull production type -------------------
def extract_ptype(col):
    m = re.search(r"(?:pp|fp)_(.+)", col)   # grabs text after pp_ / fp_
    return m.group(1) if m else "unknown"

for _df in (capex_df, opex_df, production_df):
    _df["prod_type"] = _df["variable"].apply(extract_ptype)

# rename columns for clarity
capex_df = capex_df.rename(columns={"value": "usd"})
opex_df  = opex_df.rename(columns={"value": "usd"})
production_df = production_df.rename(columns={"value": "production"})

In [29]:
# ---------------- cost aggregation ------------------------------
cost_long = pd.concat([capex_df, opex_df], ignore_index=True)

annual_cost = (
    cost_long.groupby([ "time_period", "prod_type", "cost_type"], as_index=False)["usd"].sum()
             .pivot(index=[ "time_period", "prod_type"],
                    columns="cost_type", values="usd")
             .fillna(0)
             .reset_index()
)
annual_cost["total_usd"] = annual_cost["capex"] + annual_cost["opex"]

# ---------------- production aggregation ------------------------
annual_prod = (
    production_df.groupby([ "time_period", "prod_type"], as_index=False)["production"].sum()
)

# ---------------- merge cost + production -----------------------
annual_pt = annual_cost.merge(
    annual_prod, on=[ "time_period", "prod_type"], how="left"
).fillna({"production": 0})

print("annual_pt sample:")
display(annual_pt.head())


annual_pt sample:


C:\Users\pkane\AppData\Local\Temp\ipykernel_18828\2453675818.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0)
C:\Users\pkane\AppData\Local\Temp\ipykernel_18828\2453675818.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna({"production": 0})


,time_period,prod_type,capex,opex,total_usd,production
0,0,ammonia_production,0.000000,0.380257,0.380257,0.000000
1,0,biogas,0.000000,0.000000,0.000000,0.000000
2,0,biomass,662.234195,181.414361,843.648557,9.716709
3,0,coal,32478.311760,1631.004509,34109.316269,203.994027
4,0,coal_ccs,0.000000,0.000000,0.000000,0.000000


In [30]:
annual_reg_cost = (
    cost_long.groupby(["time_period", "cost_type"], as_index=False)["usd"].sum()
        .pivot(index=["time_period"], columns="cost_type", values="usd")
        .fillna(0)
        .reset_index()
)
annual_reg_cost["total_usd"] = annual_reg_cost["capex"] + annual_reg_cost["opex"]

annual_reg_prod = (
    production_df.groupby(["time_period"], as_index=False)["production"].sum()
)

annual_reg = annual_reg_cost.merge(
    annual_reg_prod, on=[ "time_period"], how="left"
).fillna({"production": 0})

C:\Users\pkane\AppData\Local\Temp\ipykernel_18828\866425094.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0)
C:\Users\pkane\AppData\Local\Temp\ipykernel_18828\866425094.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna({"production": 0})


In [31]:
annual_pt.to_csv(OUTPUT_DIR/f"baseline_costs_and_production_by_prodtype_{OUTPUT_SUFFIX}.csv", index=False)
annual_reg.to_csv(OUTPUT_DIR/f"baseline_costs_and_production_timeseries_{OUTPUT_SUFFIX}.csv", index=False)
print("✓ CSVs with CAPEX, OPEX, total cost **and production** written.")

# -------- optional NPV (cost only) ------------------------------
DISCOUNT_RATE = 0.07
BASE_YEAR     = 2015

annual_reg["year"] = BASE_YEAR + annual_reg["time_period"]
annual_reg["dfactor"] = 1 / (1 + DISCOUNT_RATE) ** (annual_reg["year"] - BASE_YEAR)
annual_reg["npv_usd"] = annual_reg["total_usd"] * annual_reg["dfactor"]



✓ CSVs with CAPEX, OPEX, total cost **and production** written.


# Emissions

In [32]:
base_case = louisiana[louisiana['primary_id'] == 0].copy()
emissions_col = [c for c in base_case.columns 
           if 'emission_co2e' in c]
emissions_col = [c for c in emissions_col if not 'lsmm' in c]
emissions_col = [c for c in emissions_col if not 'agrc' in c]
emissions_col = [c for c in emissions_col if not 'frst' in c]
emissions_col = [c for c in emissions_col if not 'lvst' in c]
emissions_col = [c for c in emissions_col if not 'trww' in c]
emissions_col = [c for c in emissions_col if not 'waso' in c]
emissions_col = [c for c in emissions_col if not 'soil' in c]
emissions_col = [c for c in emissions_col if not 'lndu' in c]


In [33]:
emission_data = base_case[['primary_id', 'time_period']+emissions_col]
emission_data.to_csv(OUTPUT_DIR/f"emissions_data_{OUTPUT_SUFFIX}.csv", index=False)